In [6]:
from pathlib import Path
import pandas as pd

# Define dataset path
DATA_DIR = Path(
    r"C:\Users\bucco\Desktop\Laboratory of Data Science"
    r"\project\code_project\data\Project_Data"
)

QUOTES_PATH = DATA_DIR / "quotes.json"

# Verify that the file exists
if not QUOTES_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {QUOTES_PATH}"
    )

# Load original dataset
df_quotes_raw = pd.read_json(QUOTES_PATH)

# Create cleaning working copy
df_quotes_clean = df_quotes_raw.copy()

# Convert internal identifiers
id_columns = [
    "quoted_user",
    "quoting_user",
    "quoting_truth"
]

for col in id_columns:
    df_quotes_clean[col] = pd.to_numeric(
        df_quotes_clean[col],
        errors="coerce"
    ).astype("Int64")

# Preserve external IDs as strings
df_quotes_clean["quoted_truth_external_id"] = (
    df_quotes_clean["quoted_truth_external_id"]
    .astype("string")
    .str.strip()
)

# Convert timestamps
df_quotes_clean["timestamp"] = pd.to_datetime(
    df_quotes_clean["timestamp"],
    errors="coerce",
    utc=True
)

# Verify loading
print("QUOTES DATASET LOADED")
print("-" * 40)

print("Original shape:", df_quotes_raw.shape)
print("Cleaned shape:", df_quotes_clean.shape)

print("\nData types:")
display(df_quotes_clean.dtypes.to_frame("dtype"))

print("\nDataset preview:")
display(df_quotes_clean.head())

assert df_quotes_clean.shape == (10508, 5)

QUOTES DATASET LOADED
----------------------------------------
Original shape: (10508, 5)
Cleaned shape: (10508, 5)

Data types:


,dtype
timestamp,"datetime64[us, UTC]"
quoted_user,Int64
quoting_user,Int64
quoting_truth,Int64
quoted_truth_external_id,string



Dataset preview:


,timestamp,quoted_user,quoting_user,quoting_truth,quoted_truth_external_id
0,2022-07-22 09:43:00+00:00,125,131,16333,108688461672926241
1,2022-08-20 18:00:00+00:00,300,300,14949,108857444588213255
2,2022-07-14 21:13:00+00:00,2106,8212,367606,108648254976307349
3,2022-09-12 22:40:00+00:00,933,27,1593,108790130246424349
4,2022-11-11 00:00:00+00:00,134003,6657,763581,109324987069121051


In [7]:
# Define internal identifier columns
id_columns = [
    "quoted_user",
    "quoting_user",
    "quoting_truth"
]

# Convert internal identifiers to nullable integers
for col in id_columns:
    df_quotes_clean[col] = pd.to_numeric(
        df_quotes_clean[col],
        errors="coerce"
    ).astype("Int64")

# Preserve external IDs as strings
df_quotes_clean["quoted_truth_external_id"] = (
    df_quotes_clean["quoted_truth_external_id"]
    .astype("string")
    .str.strip()
)

# Validate external ID format
invalid_external_ids = (
    ~df_quotes_clean["quoted_truth_external_id"]
    .str.fullmatch(r"[0-9]+")
    .fillna(False)
).sum()

# Check missing identifiers
missing_ids = (
    df_quotes_clean[
        id_columns + ["quoted_truth_external_id"]
    ].isna().sum().sum()
)

# Check non-positive internal identifiers
non_positive_ids = (
    df_quotes_clean[id_columns]
    .le(0)
    .sum()
    .sum()
)

# Convert timestamps
df_quotes_clean["timestamp"] = pd.to_datetime(
    df_quotes_clean["timestamp"],
    errors="coerce",
    utc=True
)

# Validate timestamps
invalid_timestamps = (
    df_quotes_clean["timestamp"].isna().sum()
)

future_timestamps = (
    df_quotes_clean["timestamp"]
    > pd.Timestamp.now(tz="UTC")
).sum()

# Build validation report
identifier_validation = pd.DataFrame({
    "Check": [
        "Missing identifiers",
        "Non-positive internal IDs",
        "Invalid external ID formats",
        "Missing or invalid timestamps",
        "Future timestamps"
    ],
    "Count": [
        missing_ids,
        non_positive_ids,
        invalid_external_ids,
        invalid_timestamps,
        future_timestamps
    ]
})

print("IDENTIFIER AND TIMESTAMP VALIDATION")
print("-" * 45)

display(identifier_validation)

print("\nData types after conversion:")
display(df_quotes_clean.dtypes.to_frame("dtype"))

print("\nDataset shape:")
print(df_quotes_clean.shape)

IDENTIFIER AND TIMESTAMP VALIDATION
---------------------------------------------


,Check,Count
0,Missing identifiers,0
1,Non-positive internal IDs,0
2,Invalid external ID formats,0
3,Missing or invalid timestamps,0
4,Future timestamps,0



Data types after conversion:


,dtype
timestamp,"datetime64[us, UTC]"
quoted_user,Int64
quoting_user,Int64
quoting_truth,Int64
quoted_truth_external_id,string



Dataset shape:
(10508, 5)


In [8]:
# Check exact duplicate records
exact_duplicates = df_quotes_clean.duplicated().sum()

# Check duplicate quoting post identifiers
duplicate_quoting_truths = (
    df_quotes_clean["quoting_truth"]
    .duplicated()
    .sum()
)

# Identify self-quote relationships
self_quotes = (
    df_quotes_clean["quoted_user"]
    == df_quotes_clean["quoting_user"]
).sum()

# Check whether a quoted post is assigned to multiple users
users_per_quoted_post = (
    df_quotes_clean
    .groupby("quoted_truth_external_id")["quoted_user"]
    .nunique()
)

conflicting_quoted_users = (
    users_per_quoted_post > 1
).sum()

# Build validation summary
logical_validation = pd.DataFrame({
    "Check": [
        "Exact duplicate records",
        "Duplicate quoting_truth IDs",
        "Self-quote relationships",
        "Quoted posts assigned to multiple users"
    ],
    "Count": [
        exact_duplicates,
        duplicate_quoting_truths,
        self_quotes,
        conflicting_quoted_users
    ]
})

print("DUPLICATE AND LOGICAL CONSISTENCY")
print("-" * 45)

display(logical_validation)

print("\nExamples of conflicting quoted-user assignments:")

conflicting_ids = users_per_quoted_post[
    users_per_quoted_post > 1
].index

display(
    df_quotes_clean.loc[
        df_quotes_clean["quoted_truth_external_id"]
        .isin(conflicting_ids),
        [
            "quoted_truth_external_id",
            "quoted_user",
            "quoting_user",
            "quoting_truth"
        ]
    ].sort_values(
        "quoted_truth_external_id"
    ).head(20)
)

DUPLICATE AND LOGICAL CONSISTENCY
---------------------------------------------


,Check,Count
0,Exact duplicate records,0
1,Duplicate quoting_truth IDs,0
2,Self-quote relationships,937
3,Quoted posts assigned to multiple users,0



Examples of conflicting quoted-user assignments:


,quoted_truth_external_id,quoted_user,quoting_user,quoting_truth


In [10]:
# Define path to cleaned Users dataset
USERS_CLEAN_PATH = DATA_DIR / "users_clean.csv"

if not USERS_CLEAN_PATH.is_file():
    raise FileNotFoundError(
        f"File not found: {USERS_CLEAN_PATH}"
    )

# Load user identifiers
df_users_ids = pd.read_csv(
    USERS_CLEAN_PATH,
    usecols=["id_user"],
    dtype={"id_user": "Int64"}
)

# Extract unique valid user IDs
valid_user_ids = pd.Index(
    df_users_ids["id_user"].dropna().unique()
)

# Check foreign key references
quoted_user_exists = (
    df_quotes_clean["quoted_user"].isin(valid_user_ids)
)

quoting_user_exists = (
    df_quotes_clean["quoting_user"].isin(valid_user_ids)
)

# Identify unresolved relationships
unmatched_quotes = (
    ~quoted_user_exists | ~quoting_user_exists
)

# Count distinct unknown users
unknown_user_ids = set(
    df_quotes_clean.loc[
        ~quoted_user_exists, "quoted_user"
    ]
).union(
    set(
        df_quotes_clean.loc[
            ~quoting_user_exists, "quoting_user"
        ]
    )
)

# Build referential integrity report
referential_summary = pd.DataFrame({
    "Check": [
        "Total quote relationships",
        "Unknown quoted_user references",
        "Unknown quoting_user references",
        "Relationships with unknown users",
        "Distinct unknown user IDs"
    ],
    "Count": [
        len(df_quotes_clean),
        (~quoted_user_exists).sum(),
        (~quoting_user_exists).sum(),
        unmatched_quotes.sum(),
        len(unknown_user_ids)
    ]
})

print("REFERENTIAL INTEGRITY — USERS")
print("-" * 45)

display(referential_summary)

print("\nUnknown user IDs:")
print(sorted(unknown_user_ids)[:20])

print("\nExamples of unmatched quotes:")

display(
    df_quotes_clean.loc[
        unmatched_quotes,
        [
            "quoted_user",
            "quoting_user",
            "quoting_truth",
            "quoted_truth_external_id"
        ]
    ].head(20)
)

REFERENTIAL INTEGRITY — USERS
---------------------------------------------


,Check,Count
0,Total quote relationships,10508
1,Unknown quoted_user references,0
2,Unknown quoting_user references,9
3,Relationships with unknown users,9
4,Distinct unknown user IDs,1



Unknown user IDs:
[np.int64(1173)]

Examples of unmatched quotes:


,quoted_user,quoting_user,quoting_truth,quoted_truth_external_id
3332,24290,1173,339816,108941007530787554
3333,24290,1173,339840,108731371095118518
3334,24290,1173,339860,108626744803302358
3335,24290,1173,339871,108545897605877523
3407,24290,1173,339847,108676688786534610
3408,24290,1173,339869,108551305075906138
3409,24290,1173,339879,108545897605877523
3416,1118,1173,339904,108535682795630813
3457,695,1173,339864,108615548455606160


In [11]:
# Load cleaned Truths dataset
TRUTHS_CLEAN_PATH = DATA_DIR / "truths_clean.csv"

df_truths_ref = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    usecols=["id", "author"],
    dtype={
        "id": "Int64",
        "author": "string"
    },
    low_memory=False
)

# Normalize author identifiers
df_truths_ref["author"] = pd.to_numeric(
    df_truths_ref["author"],
    errors="coerce"
).astype("Int64")

# Prepare quoting post references
df_quoting_posts = df_quotes_clean.merge(
    df_truths_ref.rename(
        columns={
            "id": "quoting_truth",
            "author": "actual_author"
        }
    ),
    on="quoting_truth",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Identify matched and unmatched posts
matched_posts = (
    df_quoting_posts["_merge"] == "both"
)

unmatched_posts = ~matched_posts

# Check author consistency only for matched posts
comparable_authors = (
    matched_posts
    & df_quoting_posts["actual_author"].notna()
)

author_mismatches = (
    comparable_authors
    & df_quoting_posts["quoting_user"].ne(
        df_quoting_posts["actual_author"]
    ).fillna(False)
)

# Build validation summary
truths_validation = pd.DataFrame({
    "Check": [
        "Total quote relationships",
        "Matched quoting posts",
        "Unmatched quoting posts",
        "Matched posts with available authors",
        "Quoting-user / author mismatches"
    ],
    "Count": [
        len(df_quoting_posts),
        matched_posts.sum(),
        unmatched_posts.sum(),
        comparable_authors.sum(),
        author_mismatches.sum()
    ]
})

print("REFERENTIAL INTEGRITY — QUOTING POSTS")
print("-" * 45)

display(truths_validation)

print("\nAuthor inconsistencies:")

display(
    df_quoting_posts.loc[
        author_mismatches,
        [
            "quoting_truth",
            "quoting_user",
            "actual_author"
        ]
    ].head(20)
)

# Investigate the missing user 1173
print("\nQUOTING RELATIONSHIPS FOR USER 1173")

display(
    df_quoting_posts.loc[
        df_quoting_posts["quoting_user"] == 1173,
        [
            "quoting_truth",
            "quoting_user",
            "actual_author",
            "_merge"
        ]
    ]
)

REFERENTIAL INTEGRITY — QUOTING POSTS
---------------------------------------------


,Check,Count
0,Total quote relationships,10508
1,Matched quoting posts,9594
2,Unmatched quoting posts,914
3,Matched posts with available authors,9594
4,Quoting-user / author mismatches,0



Author inconsistencies:


,quoting_truth,quoting_user,actual_author



QUOTING RELATIONSHIPS FOR USER 1173


,quoting_truth,quoting_user,actual_author,_merge
3332,339816,1173,1173,both
3333,339840,1173,1173,both
3334,339860,1173,1173,both
3335,339871,1173,1173,both
3407,339847,1173,1173,both
3408,339869,1173,1173,both
3409,339879,1173,1173,both
3416,339904,1173,1173,both
3457,339864,1173,1173,both


In [12]:
# Load the relevant attributes from the main Truths dataset
df_quoted_posts_ref = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    usecols=["id", "external_id", "author"],
    dtype={
        "id": "Int64",
        "external_id": "string",
        "author": "string"
    },
    low_memory=False
)

# Normalize external IDs without floating-point conversion
df_quoted_posts_ref["external_id"] = (
    df_quoted_posts_ref["external_id"]
    .str.strip()
)

# Convert authors to nullable integers
df_quoted_posts_ref["author"] = pd.to_numeric(
    df_quoted_posts_ref["author"],
    errors="coerce"
).astype("Int64")

# Exclude missing or invalid external IDs
valid_external_mask = (
    df_quoted_posts_ref["external_id"]
    .str.fullmatch(r"[0-9]+")
    .fillna(False)
)

df_quoted_posts_ref = df_quoted_posts_ref.loc[
    valid_external_mask
].copy()

# Check whether external IDs are unique
duplicate_external_ids = (
    df_quoted_posts_ref["external_id"]
    .duplicated()
    .sum()
)

print("Duplicate external IDs in Truths:",
      duplicate_external_ids)

# Identify quoted posts with matching external IDs
quoted_matches = df_quotes_clean[
    "quoted_truth_external_id"
].isin(
    df_quoted_posts_ref["external_id"]
)

print("\nQUOTED POST REFERENTIAL INTEGRITY")
print("-" * 45)

print("Total quote relationships:",
      len(df_quotes_clean))

print("Matched quoted posts:",
      quoted_matches.sum())

print("Unmatched quoted posts:",
      (~quoted_matches).sum())

print("\nMatching percentage:",
      round(quoted_matches.mean() * 100, 2),
      "%"
)

# Verify that external IDs define a unique mapping
# before comparing quoted users with post authors
if duplicate_external_ids == 0:

    df_quoted_validation = df_quotes_clean.merge(
        df_quoted_posts_ref.rename(
            columns={
                "id": "quoted_internal_id",
                "external_id": "quoted_truth_external_id",
                "author": "actual_quoted_author"
            }
        ),
        on="quoted_truth_external_id",
        how="left",
        validate="many_to_one",
        indicator=True
    )

    # Compare authors only when post and author are available
    comparable = (
        df_quoted_validation["_merge"].eq("both")
        & df_quoted_validation["actual_quoted_author"].notna()
    )

    author_mismatches = (
        comparable
        & df_quoted_validation["quoted_user"].ne(
            df_quoted_validation["actual_quoted_author"]
        ).fillna(False)
    )

    print("\nMatched posts with comparable authors:",
          comparable.sum())

    print("Quoted-user / author mismatches:",
          author_mismatches.sum())

    print("\nExamples of author mismatches:")

    display(
        df_quoted_validation.loc[
            author_mismatches,
            [
                "quoted_truth_external_id",
                "quoted_user",
                "actual_quoted_author",
                "quoted_internal_id"
            ]
        ].head(20)
    )

else:
    print(
        "\nAuthor comparison deferred: "
        "external IDs are not unique."
    )

Duplicate external IDs in Truths: 145944

QUOTED POST REFERENTIAL INTEGRITY
---------------------------------------------
Total quote relationships: 10508
Matched quoted posts: 2041
Unmatched quoted posts: 8467

Matching percentage: 19.42 %

Author comparison deferred: external IDs are not unique.


In [13]:
# Group Truths records by external post identifier
author_mapping = (
    df_quoted_posts_ref
    .groupby("external_id")
    .agg(
        matching_truth_records=("id", "size"),
        distinct_authors=("author", "nunique"),
        unique_author=("author", "first"),
        missing_authors=("author", lambda x: x.isna().sum())
    )
    .reset_index()
)

# Retain only unambiguous mappings
author_mapping["author_is_unambiguous"] = (
    author_mapping["distinct_authors"].eq(1)
    & author_mapping["missing_authors"].eq(0)
)

# Match quote references to the grouped mapping
df_quoted_author_check = df_quotes_clean.merge(
    author_mapping,
    left_on="quoted_truth_external_id",
    right_on="external_id",
    how="left",
    validate="many_to_one"
)

# Identify matched references
matched = (
    df_quoted_author_check["external_id"].notna()
)

# Identify references with reliable author mappings
comparable = (
    matched
    & df_quoted_author_check["author_is_unambiguous"]
      .fillna(False)
)

# Compare quoted users with known authors
author_mismatches = (
    comparable
    & df_quoted_author_check["quoted_user"].ne(
        df_quoted_author_check["unique_author"]
    ).fillna(False)
)

# Summarize results
author_consistency_report = pd.DataFrame({
    "Check": [
        "Total quote relationships",
        "Matched quoted post references",
        "References with unambiguous authors",
        "Matched references with ambiguous authors",
        "Quoted-user / author mismatches"
    ],
    "Count": [
        len(df_quotes_clean),
        matched.sum(),
        comparable.sum(),
        (matched & ~comparable).sum(),
        author_mismatches.sum()
    ]
})

print("QUOTED AUTHOR CONSISTENCY")
print("-" * 45)

display(author_consistency_report)

print("\nExamples of author mismatches:")

display(
    df_quoted_author_check.loc[
        author_mismatches,
        [
            "quoted_truth_external_id",
            "quoted_user",
            "unique_author",
            "matching_truth_records"
        ]
    ].head(20)
)

print("\nExamples of ambiguous references:")

display(
    df_quoted_author_check.loc[
        matched & ~comparable,
        [
            "quoted_truth_external_id",
            "matching_truth_records",
            "distinct_authors",
            "missing_authors"
        ]
    ].head(20)
)

QUOTED AUTHOR CONSISTENCY
---------------------------------------------


,Check,Count
0,Total quote relationships,10508
1,Matched quoted post references,2041
2,References with unambiguous authors,2041
3,Matched references with ambiguous authors,0
4,Quoted-user / author mismatches,6



Examples of author mismatches:


,quoted_truth_external_id,quoted_user,unique_author,matching_truth_records
2824,108749044759354785,38288,97,1
3111,108749044759354785,38288,97,1
3332,108941007530787554,24290,1173,1
3335,108545897605877523,24290,1173,1
3408,108551305075906138,24290,1173,1
3409,108545897605877523,24290,1173,1



Examples of ambiguous references:


,quoted_truth_external_id,matching_truth_records,distinct_authors,missing_authors


In [14]:
# Identify the six author mismatches
mismatch_ids = df_quoted_author_check.loc[
    author_mismatches,
    "quoted_truth_external_id"
].unique()

# Load relevant post information from Truths
df_truths_details = pd.read_csv(
    TRUTHS_CLEAN_PATH,
    usecols=[
        "id",
        "external_id",
        "author",
        "url",
        "timestamp"
    ],
    dtype={
        "id": "Int64",
        "external_id": "string",
        "author": "string",
        "url": "string"
    },
    low_memory=False
)

# Normalize identifiers
df_truths_details["external_id"] = (
    df_truths_details["external_id"].str.strip()
)

df_truths_details["author"] = pd.to_numeric(
    df_truths_details["author"],
    errors="coerce"
).astype("Int64")

# Select posts involved in mismatches
mismatch_posts = df_truths_details.loc[
    df_truths_details["external_id"].isin(
        mismatch_ids
    )
].copy()

# Extract usernames from post URLs
mismatch_posts["username_from_url"] = (
    mismatch_posts["url"]
    .str.extract(
        r"(?i)truthsocial\.com/@([^/]+)/posts/",
        expand=False
    )
)

print("QUOTED AUTHOR MISMATCH INVESTIGATION")
print("-" * 45)

print("\nMatching post details:")
display(
    mismatch_posts[
        [
            "id",
            "external_id",
            "author",
            "username_from_url",
            "url",
            "timestamp"
        ]
    ]
)

print("\nQuote relationship details:")
display(
    df_quoted_author_check.loc[
        author_mismatches,
        [
            "quoted_truth_external_id",
            "quoted_user",
            "quoting_user",
            "quoting_truth",
            "unique_author"
        ]
    ]
)

QUOTED AUTHOR MISMATCH INVESTIGATION
---------------------------------------------

Matching post details:


,id,external_id,author,username_from_url,url,timestamp
462635,336090,108749044759354785,97,cpac,https://truthsocial.com/@cpac/posts/1087490447...,2022-08-01 14:32:00
465762,339818,108941007530787554,1173,AP,https://truthsocial.com/@AP/posts/108941007530...,2022-09-04 12:10:00
465797,339871,108551305075906138,1173,AP,https://truthsocial.com/@AP/posts/108551305075...,2022-06-27 16:24:00
465821,339882,108545897605877523,1173,AP,https://truthsocial.com/@AP/posts/108545897605...,2022-06-26 17:29:00



Quote relationship details:


,quoted_truth_external_id,quoted_user,quoting_user,quoting_truth,unique_author
2824,108749044759354785,38288,351,318938,97
3111,108749044759354785,38288,788,76637,97
3332,108941007530787554,24290,1173,339816,1173
3335,108545897605877523,24290,1173,339871,1173
3408,108551305075906138,24290,1173,339869,1173
3409,108545897605877523,24290,1173,339879,1173
